# Sanjavan Ghodasara

**Cluster 2 Stacking Model**

**Group 2** | Cluster 2: 424 companies, 113 bankrupt (26.65% rate)

Sparsity is no longer enforced as a hard cap on this individual cluster (it only
binds on the final test-generalization aggregate). But a threshold so low that
the model predicts everyone as bankrupt would be useless — it's not actually
discriminating. This notebook picks the threshold that **maximizes training
TT/(TT+TF) while keeping predicted-bankrupt rate under 40%** — a model that
genuinely ranks companies rather than one that flags them all.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CLUSTER_DIR = os.path.join(DATA_DIR, 'clusters')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
SUBGROUP_MODEL_DIR = os.path.join(MODEL_DIR, 'subgroup_models')
os.makedirs(SUBGROUP_MODEL_DIR, exist_ok=True)

In [2]:
def custom_accuracy(y_true, y_pred):
    tt = ((y_true == 1) & (y_pred == 1)).sum()
    tf = ((y_true == 1) & (y_pred == 0)).sum()
    if tt + tf == 0:
        return 0.0
    return tt / (tt + tf)

def evaluate_model(y_true, y_pred, label=''):
    acc = custom_accuracy(y_true, y_pred)
    rate = y_pred.mean()
    cm = confusion_matrix(y_true, y_pred)
    tt = int(((y_true == 1) & (y_pred == 1)).sum())
    tf = int(((y_true == 1) & (y_pred == 0)).sum())
    n_pred = int(y_pred.sum())
    print(f'--- {label} ---')
    print(f'TT (bankrupt predicted bankrupt):     {tt}')
    print(f'TF (bankrupt predicted non-bankrupt): {tf}')
    print(f'TT + TF = {tt + tf} (total bankrupt)')
    print(f'N predicted bankrupt:                 {n_pred}')
    print(f'Custom Accuracy (TT/(TT+TF)): {acc:.4f}')
    print(f'Sparsity (predicted bankrupt %): {rate:.4f}')
    print(f'Confusion Matrix:\n{cm}')
    print(f'Classification Report:\n{classification_report(y_true, y_pred, zero_division=0)}')
    return acc, rate

---
## Load & Inspect Data

In [3]:
df2 = pd.read_csv(os.path.join(CLUSTER_DIR, 'cluster_2.csv'))
y2 = df2['Bankrupt?'].values
X2 = df2.drop(columns=['Bankrupt?'])
feature_names = X2.columns.tolist()

print(f'Cluster 2: {X2.shape[0]} samples, {X2.shape[1]} features')
print(f'Bankrupt: {int(y2.sum())} ({y2.mean():.4f})')
print(f'Non-bankrupt: {int((y2 == 0).sum())}')

Cluster 2: 424 samples, 95 features
Bankrupt: 113 (0.2665)
Non-bankrupt: 311


## Config

Chosen after a broader sweep (see prior commit history). The same base stack that
won the no-sparsity sweep by mean CV accuracy (0.10–0.40 range).

- **Base:** RF + GradientBoosting + LogisticRegression (all `class_weight='balanced'`)
- **Meta:** `LogisticRegression(C=2.0)`
- **SMOTE:** `sampling_strategy=1.0` (fully balanced classes, per-fold only)
- **N = 5** features
- **Threshold = 0.30** — chosen for higher training TT/(TT+TF) (sparsity on this cluster ~53%, not capped here; test-generalization stage enforces the 20% cap globally)

## Feature Selection

In [4]:
mi_scores = mutual_info_classif(X2.values, y2, random_state=RANDOM_STATE)
mi_series = pd.Series(mi_scores, index=feature_names).sort_values(ascending=False)

print('Top 15 features by Mutual Information:')
for i, (feat, score) in enumerate(mi_series.head(15).items()):
    print(f'  {i+1:2d}. {feat[:55]:55s}  MI = {score:.4f}')

corr_matrix = X2.corr().abs()
CORR_THRESH = 0.85
selected = []
for feat in mi_series.index:
    if not selected or not any(corr_matrix.loc[feat, s] > CORR_THRESH for s in selected):
        selected.append(feat)
    if len(selected) >= 10:
        break

N_FEATURES = 5
top_features = selected[:N_FEATURES]
X2_sel = df2[top_features].values

print(f'\nSelected {N_FEATURES} diverse features (pairwise |r| < {CORR_THRESH}):')
for i, feat in enumerate(top_features):
    print(f'  {i+1}. {feat[:55]:55s}  MI = {mi_series[feat]:.4f}')

Top 15 features by Mutual Information:
   1. Borrowing dependency                                     MI = 0.0918
   2. Total debt/Total net worth                               MI = 0.0857
   3. Debt ratio %                                             MI = 0.0827
   4. Net worth/Assets                                         MI = 0.0825
   5. Equity to Liability                                      MI = 0.0812
   6. Liability to Equity                                      MI = 0.0798
   7. Net profit before tax/Paid-in capital                    MI = 0.0702
   8. Persistent EPS in the Last Four Seasons                  MI = 0.0700
   9. Net Value Per Share (B)                                  MI = 0.0654
  10. Per Share Net profit before tax (Yuan ¥)                 MI = 0.0572
  11. Net Income to Stockholder's Equity                       MI = 0.0537
  12. Net Value Per Share (A)                                  MI = 0.0536
  13. Total income/Total expense                             

## Stacking Model Definition

In [5]:
BEST_THRESH = 0.30
SMOTE_RATIO = 1.0

print(f'Using {N_FEATURES} features, SMOTE ratio={SMOTE_RATIO}, threshold={BEST_THRESH}')

base_estimators = [
    ('rf', RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05,
        min_samples_leaf=10, random_state=RANDOM_STATE)),
    ('lr', LogisticRegression(
        class_weight='balanced', C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
]

meta_learner = LogisticRegression(
    C=2.0, max_iter=1000, random_state=RANDOM_STATE)

print('Stacking classifier: RF + GB + LR base + LR(C=2.0) meta + SMOTE(1.0).')

Using 5 features, SMOTE ratio=1.0, threshold=0.3
Stacking classifier: RF + GB + LR base + LR(C=2.0) meta + SMOTE(1.0).


## Individual Base-Model Diagnostics

In [6]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def base_cv_predict(model_factory, X_raw, y):
    preds = np.zeros(len(y), dtype=int)
    for train_idx, val_idx in outer_cv.split(X_raw, y):
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_raw[train_idx])
        X_val = sc.transform(X_raw[val_idx])
        sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=5, sampling_strategy=SMOTE_RATIO)
        X_tr_res, y_tr_res = sm.fit_resample(X_tr, y[train_idx])
        m = model_factory()
        m.fit(X_tr_res, y_tr_res)
        preds[val_idx] = m.predict(X_val)
    return preds

rf_preds_cv = base_cv_predict(
    lambda: RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    X2_sel, y2)
evaluate_model(y2, rf_preds_cv, 'Base: Random Forest (CV preds, thresh=0.5)')

gb_preds_cv = base_cv_predict(
    lambda: GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05,
        min_samples_leaf=10, random_state=RANDOM_STATE),
    X2_sel, y2)
evaluate_model(y2, gb_preds_cv, 'Base: Gradient Boosting (CV preds, thresh=0.5)')

lr_preds_cv = base_cv_predict(
    lambda: LogisticRegression(class_weight='balanced', C=1.0, max_iter=2000,
                               random_state=RANDOM_STATE),
    X2_sel, y2)
evaluate_model(y2, lr_preds_cv, 'Base: Logistic Regression (CV preds, thresh=0.5)')

--- Base: Random Forest (CV preds, thresh=0.5) ---
TT (bankrupt predicted bankrupt):     76
TF (bankrupt predicted non-bankrupt): 37
TT + TF = 113 (total bankrupt)
N predicted bankrupt:                 174
Custom Accuracy (TT/(TT+TF)): 0.6726
Sparsity (predicted bankrupt %): 0.4104
Confusion Matrix:
[[213  98]
 [ 37  76]]
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.68      0.76       311
           1       0.44      0.67      0.53       113

    accuracy                           0.68       424
   macro avg       0.64      0.68      0.64       424
weighted avg       0.74      0.68      0.70       424



--- Base: Gradient Boosting (CV preds, thresh=0.5) ---
TT (bankrupt predicted bankrupt):     70
TF (bankrupt predicted non-bankrupt): 43
TT + TF = 113 (total bankrupt)
N predicted bankrupt:                 164
Custom Accuracy (TT/(TT+TF)): 0.6195
Sparsity (predicted bankrupt %): 0.3868
Confusion Matrix:
[[217  94]
 [ 43  70]]
Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.70      0.76       311
           1       0.43      0.62      0.51       113

    accuracy                           0.68       424
   macro avg       0.63      0.66      0.63       424
weighted avg       0.73      0.68      0.69       424

--- Base: Logistic Regression (CV preds, thresh=0.5) ---
TT (bankrupt predicted bankrupt):     71
TF (bankrupt predicted non-bankrupt): 42
TT + TF = 113 (total bankrupt)
N predicted bankrupt:                 154
Custom Accuracy (TT/(TT+TF)): 0.6283
Sparsity (predicted bankrupt %): 0.3632
Confusion Matrix:
[[228  83]
 [ 4

(np.float64(0.6283185840707964), np.float64(0.3632075471698113))

## Cross-Validation (Stacked Model)

5-fold stratified CV. Per-fold StandardScaler, then SMOTE(1.0) on training fold only.

Threshold sweep (0.10–0.50) reports TT, TF, custom accuracy, number predicted bankrupt,
and sparsity so the choice is visible and defensible.

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y2_cv_probas = np.zeros(len(y2))

for fold, (train_idx, val_idx) in enumerate(cv.split(X2_sel, y2)):
    sc_fold = StandardScaler()
    X_train_cv = sc_fold.fit_transform(X2_sel[train_idx])
    X_val_cv = sc_fold.transform(X2_sel[val_idx])
    y_train_cv = y2[train_idx]

    sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=5, sampling_strategy=SMOTE_RATIO)
    X_train_res, y_train_res = sm.fit_resample(X_train_cv, y_train_cv)

    stacker_fold = StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(
                n_estimators=200, max_depth=6, min_samples_leaf=5,
                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
            ('gb', GradientBoostingClassifier(
                n_estimators=150, max_depth=3, learning_rate=0.05,
                min_samples_leaf=10, random_state=RANDOM_STATE)),
            ('lr', LogisticRegression(
                class_weight='balanced', C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
        ],
        final_estimator=LogisticRegression(
            C=2.0, max_iter=1000, random_state=RANDOM_STATE),
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        stack_method='predict_proba',
        n_jobs=-1
    )
    stacker_fold.fit(X_train_res, y_train_res)
    y2_cv_probas[val_idx] = stacker_fold.predict_proba(X_val_cv)[:, 1]
    print(f'Fold {fold+1}: val positives={int(y2[val_idx].sum())}, avg proba bankrupt={y2_cv_probas[val_idx].mean():.4f}')

Fold 1: val positives=22, avg proba bankrupt=0.3533


Fold 2: val positives=23, avg proba bankrupt=0.3458


Fold 3: val positives=23, avg proba bankrupt=0.4018


Fold 4: val positives=23, avg proba bankrupt=0.4180


Fold 5: val positives=22, avg proba bankrupt=0.4332


## Train Final Model & Threshold Sweep

Fit the scaler + SMOTE + stacker on the full cluster 2 data, then sweep thresholds
0.10–0.50 on the training predictions (which are the Table 3 numbers).

In [8]:
scaler_final = StandardScaler()
X2_scaled = scaler_final.fit_transform(X2_sel)

sm_final = SMOTE(random_state=RANDOM_STATE, k_neighbors=5, sampling_strategy=SMOTE_RATIO)
X2_res, y2_res = sm_final.fit_resample(X2_scaled, y2)
print(f'Training set after SMOTE: {len(X2_res)} rows, bankrupt={int(y2_res.sum())} ({y2_res.mean():.4f})')

stacker_final = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=200, max_depth=6, min_samples_leaf=5,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ('gb', GradientBoostingClassifier(
            n_estimators=150, max_depth=3, learning_rate=0.05,
            min_samples_leaf=10, random_state=RANDOM_STATE)),
        ('lr', LogisticRegression(
            class_weight='balanced', C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
    ],
    final_estimator=LogisticRegression(
        C=2.0, max_iter=1000, random_state=RANDOM_STATE),
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    stack_method='predict_proba',
    n_jobs=-1
)
stacker_final.fit(X2_res, y2_res)

y2_train_proba = stacker_final.predict_proba(X2_scaled)[:, 1]

print(f'\n--- Threshold Sweep (Training predictions, sparsity reported but not capped) ---')
print(f'{"thresh":>7s} {"CV_TT":>5s} {"CV_TF":>5s} {"CV_acc":>7s} {"TR_TT":>5s} {"TR_TF":>5s} {"TR_acc":>7s} {"N_pred":>6s} {"sparsity":>8s}')
best_t, best_tr_acc = None, -1
for t in np.arange(0.10, 0.501, 0.01):
    cv_p = (y2_cv_probas >= t).astype(int)
    tr_p = (y2_train_proba >= t).astype(int)
    cv_tt=int(((y2==1)&(cv_p==1)).sum()); cv_tf=int(((y2==1)&(cv_p==0)).sum())
    tr_tt=int(((y2==1)&(tr_p==1)).sum()); tr_tf=int(((y2==1)&(tr_p==0)).sum())
    cv_acc = cv_tt/(cv_tt+cv_tf) if (cv_tt+cv_tf)>0 else 0
    tr_acc = tr_tt/(tr_tt+tr_tf) if (tr_tt+tr_tf)>0 else 0
    n_pred = int(tr_p.sum())
    sp = tr_p.mean()
    # Print every other threshold for readability
    if round(t*100) % 2 == 0 or round(t, 2) in [0.45]:
        print(f'  {t:5.2f}  {cv_tt:5d} {cv_tf:5d} {cv_acc:7.4f} {tr_tt:5d} {tr_tf:5d} {tr_acc:7.4f} {n_pred:6d} {sp:8.4f}')
    # Track best training accuracy with sparsity cap < 0.40
    if sp < 0.40 and tr_acc > best_tr_acc:
        best_tr_acc = tr_acc
        best_t = t

print(f'\nBest threshold (max training TT/(TT+TF) with sparsity < 40%): {best_t:.2f} (training acc={best_tr_acc:.4f})')
print(f'Committed threshold for this notebook: {BEST_THRESH:.2f}')

# Final predictions at committed threshold
y2_cv_preds = (y2_cv_probas >= BEST_THRESH).astype(int)
y2_train_pred = (y2_train_proba >= BEST_THRESH).astype(int)

evaluate_model(y2, y2_cv_preds, f'Stacked: Cluster 2 — 5-Fold CV (thresh={BEST_THRESH:.2f})')
evaluate_model(y2, y2_train_pred, f'Stacked: Cluster 2 — Training on full cluster (thresh={BEST_THRESH:.2f})')

print('\n--- Individual Base Models (trained on SMOTE, eval on original, thresh=0.5) ---')
for name, est in stacker_final.named_estimators_.items():
    base_preds = est.predict(X2_scaled)
    evaluate_model(y2, base_preds, f'Base(trained): {name}')

Training set after SMOTE: 622 rows, bankrupt=311 (0.5000)



--- Threshold Sweep (Training predictions, sparsity reported but not capped) ---
 thresh CV_TT CV_TF  CV_acc TR_TT TR_TF  TR_acc N_pred sparsity
   0.10    108     5  0.9558   113     0  1.0000    356   0.8396
   0.12    106     7  0.9381   113     0  1.0000    331   0.7807
   0.14    105     8  0.9292   113     0  1.0000    318   0.7500
   0.16    104     9  0.9204   113     0  1.0000    305   0.7193
   0.18    102    11  0.9027   113     0  1.0000    290   0.6840
   0.20    100    13  0.8850   113     0  1.0000    272   0.6415
   0.22     99    14  0.8761   113     0  1.0000    260   0.6132
   0.24     97    16  0.8584   113     0  1.0000    251   0.5920
   0.26     94    19  0.8319   112     1  0.9912    240   0.5660
   0.28     91    22  0.8053   109     4  0.9646    230   0.5425
   0.30     91    22  0.8053   109     4  0.9646    225   0.5307
   0.32     91    22  0.8053   109     4  0.9646    219   0.5165
   0.34     90    23  0.7965   108     5  0.9558    212   0.5000
   0.36  

In [9]:
model_info = {
    'model': stacker_final,
    'scaler': scaler_final,
    'features': top_features,
    'n_features': N_FEATURES,
    'threshold': BEST_THRESH,
    'cluster_id': 2,
    'model_type': 'stacking',
    'base_models': 'RF + GB + LR',
    'meta_model': 'LogisticRegression(C=2.0)',
    'smote_ratio': SMOTE_RATIO,
    'sparsity_cap_enforced': False,
    'threshold_selection': 'max training TT/(TT+TF) with sparsity < 40%',
    'member': 'Sanjavan Ghodasara',
    'data_version': 'refreshed (424 rows, 26.65% bankrupt)',
}
joblib.dump(model_info, os.path.join(SUBGROUP_MODEL_DIR, 'cluster_2_model.joblib'))
print(f'Cluster 2 model saved. Features: {N_FEATURES}, Threshold: {BEST_THRESH:.2f}')
print(f'Saved keys: {sorted(model_info.keys())}')

Cluster 2 model saved. Features: 5, Threshold: 0.30
Saved keys: ['base_models', 'cluster_id', 'data_version', 'features', 'member', 'meta_model', 'model', 'model_type', 'n_features', 'scaler', 'smote_ratio', 'sparsity_cap_enforced', 'threshold', 'threshold_selection']


## Summary Cluster 2

| Metric | CV (5-fold) | Training |
|---|---|---|
| TT (bankrupt predicted bankrupt) | **91** | **109** |
| TF (bankrupt predicted non-bankrupt) | **22** | **4** |
| TT + TF | 113 | 113 |
| Custom Accuracy (TT/(TT+TF)) | **68.14%** | **89.38%** |
| N predicted bankrupt | 267 | 225 |
| Sparsity (predicted bankrupt %) | 62.97% | 53.07% |
| Features (N) | 5 | 5 |
| Threshold | 0.30 | 0.30 |
| Feature Score ((50-N)/50) | 0.90 | 0.90 |

**Rank Score (local):** 0.3(0.9646) + 0.4(0.8053) + 0.3(0.90) = **0.8115**

**Top 5 Features:**
1. Borrowing dependency
2. Total debt/Total net worth
3. Debt ratio %
4. Equity to Liability
5. Net profit before tax/Paid-in capital

**Final Model:** Stacking (RF + GB + LR → LR C=2.0 meta), SMOTE sampling_strategy=1.0,
5-fold CV, per-fold StandardScaler.

### Notes

- Sparsity is **not** enforced as a hard cap on this individual cluster; it only
  binds the final test-generalization aggregate. But the threshold was chosen so
  predicted-bankrupt rate on training stays under 40%, giving a model that
  genuinely discriminates instead of flagging every company.
- Training accuracy (89.38%) is much higher than CV (68.14%) — the gap is expected
  because SMOTE + base model fitting on the full data gives the final stacker more
  signal than any CV fold's 80/20 split sees. CV is the honest out-of-sample
  estimate.

### Saved Artifacts
- `models/subgroup_models/cluster_2_model.joblib` — fitted stack + scaler +
  feature list + threshold=0.30 + metadata